# Handwritten circuit reader (Digitize-HCD)
Takes a scanned hand-drawn circuit, finds every component, reads the handwritten value next to it and attaches the value to the right component.

**Pipeline**
1. **Detection:** one YOLO11m at 1024 px finds the 17 component classes *and* the handwritten text (an 18th class, `text`) in a single pass.
2. **Reading:** TrOCR (`microsoft/trocr-base-handwritten`), fine-tuned on the dataset's text crops.
3. **Matching:** each value goes to the nearest component whose class fits its unit.

**How to run (Kaggle):** attach the Digitize-HCD dataset, Accelerator = GPU T4 (one GPU, `device=0`), Internet = On (to install packages and download pretrained weights). A full run takes about 2.5 h.

**Self-contained:** everything is trained from scratch here from the Digitize-HCD dataset (pretrained YOLO11m and TrOCR weights are downloaded automatically). Nothing from earlier runs or other notebooks is needed.

**Flag (section 0):**
- `RUN_TEST`: keep `False` while developing; set `True` only for the final run. The test set is evaluated once, in Part 4, after every choice has been made on validation.

| Section | Content |
|---|---|
| 0 | Setup |
| **Part 1** | **Detection: components and text, one model** |
| 1.1 – 1.3 | Load, clean, split and convert the component and text annotations |
| 1.4 – 1.6 | Train, evaluate on validation (picks `CONF` and `TEXT_CONF`), error analysis |
| **Part 2** | **Reading: fine-tune TrOCR** |
| 2.1 – 2.2 | TrOCR zero-shot, then fine-tuned |
| **Part 3** | **Matching and full pipeline** |
| 3.1 – 3.2 | Matching rule on ground truth, full pipeline on validation |
| **Part 4** | **Final evaluation on the test set (run once)** |
| **Part 5** | **Examples and outputs** |

## 0. Setup

In [ ]:
!pip install -q ultralytics==8.4.182 transformers==4.57.6   # transformers 5.x cannot load the TrOCR tokenizer
import os, json, shutil, random, glob, datetime
import numpy as np, pandas as pd, cv2, torch, yaml
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from ultralytics import YOLO, settings
settings.update({"raytune": False})   # Kaggle has Ray installed; its callback only prints get_trial_id warnings

RUN_TEST = False         # True only for the final run: the test set is evaluated once

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

found = glob.glob("/kaggle/input/**/component_annotations.json", recursive=True)
assert found, "Digitize-HCD not found: add it with Add Input (right panel)"
ANN = found[0]
BASE = os.path.dirname(ANN)
IMG_DIR = os.path.join(BASE, "Circuit Diagram Images")
TXT_JSON = os.path.join(BASE, "text_annotations.json")

WORK = "/kaggle/working"
OUT = f"{WORK}/hcd_yolo"         # components + text in YOLO format
RUNS = f"{WORK}/runs"            # weights, metrics and figures: everything here goes into the output zip
OCR_DIR = f"{WORK}/trocr_values" # fine-tuned reader (not zipped, ~1.3 GB)
DATA_YAML = f"{OUT}/data.yaml"
RUN_NAME = "detector_yolo11m_1024"
TEST_DONE = f"{WORK}/TEST_DONE.txt"
os.makedirs(RUNS, exist_ok=True)
if RUN_TEST:
    assert not os.path.exists(TEST_DONE), "Test set already evaluated"

print(BASE)

# Part 1: Detection (components and text, one model)

## 1.1 Load and inspect the annotations
- **Components** are in COCO format (`component_annotations.json`): each box is `[x, y, w, h]` in pixels.
- **Text** is in `text_annotations.json`: each box is `[x1, y1, x2, y2]` with its transcription. Texts are short (about 3 characters): a number, a prefix and a unit, e.g. `1kΩ`, `2kHz`, `5μF`.

Both are joined on the image file name. The per-class counts show the **class imbalance** (text is by far the largest class). The box sizes justify `imgsz=1024`: YOLO resizes the longest side of each image to `imgsz`, so we compute how large the *short side* of each box becomes. At 640 px many symbols and texts get very small; at 1024 px they stay larger.

In [ ]:
stem = lambda p: os.path.splitext(os.path.basename(p))[0]
coco = json.load(open(ANN))
cats = {c["id"]: c["name"] for c in coco["categories"]}
imgs = pd.DataFrame(coco["images"]).set_index("id")
imgs["stem"] = imgs["file_name"].map(stem)
stem2id = dict(zip(imgs["stem"], imgs.index))
stem2file = {stem(f): f for f in os.listdir(IMG_DIR)}

# Components: [x, y, w, h] -> x1, y1, x2, y2
df = pd.DataFrame(coco["annotations"])
df["class"] = df["category_id"].map(cats)
df[["x", "y", "bw", "bh"]] = pd.DataFrame(df["bbox"].tolist(), index=df.index)
df["x1"], df["y1"], df["x2"], df["y2"] = df["x"], df["y"], df["x"] + df["bw"], df["y"] + df["bh"]

# Text: [x1, y1, x2, y2] + transcription, joined to the COCO images by file name
txt_json = json.load(open(TXT_JSON))["data_list"]
tx = pd.DataFrame([(stem(im["file_name"]), i["text"], *i["bbox"]) for im in txt_json for i in im["instances"]],
                  columns=["stem", "text", "x1", "y1", "x2", "y2"])
print("text boxes on images without component annotations (dropped):", int((~tx["stem"].isin(stem2id)).sum()))
tx = tx[tx["stem"].isin(stem2id)].copy()
tx["image_id"], tx["class"] = tx["stem"].map(stem2id), "text"
size_txt = {stem(im["file_name"]): (im["width"], im["height"]) for im in txt_json}
print("images whose size differs between the two files:",
      sum(size_txt.get(s, (w, h)) != (w, h) for s, w, h in zip(imgs["stem"], imgs["width"], imgs["height"])))

boxes = pd.concat([df[["image_id", "class", "x1", "y1", "x2", "y2"]],
                   tx[["image_id", "class", "text", "x1", "y1", "x2", "y2"]]], ignore_index=True)   # "text" is NaN for components
boxes["W"], boxes["H"] = boxes["image_id"].map(imgs["width"]), boxes["image_id"].map(imgs["height"])

files = os.listdir(IMG_DIR)
print("Images:", len(imgs), "| Component boxes:", len(df), "| Text boxes:", len(tx), "| Component classes:", len(cats))
print("Images missing on disk:", (~imgs["file_name"].isin(files)).sum())
print("\nInstances per class:\n", boxes["class"].value_counts().to_string())

# short side of each box after YOLO resizes the image (longest side -> imgsz)
short = np.minimum(boxes["x2"] - boxes["x1"], boxes["y2"] - boxes["y1"]) / np.maximum(boxes["W"], boxes["H"])
for s in (640, 1024):
    for name, m in (("components", boxes["class"] != "text"), ("text", boxes["class"] == "text")):
        print(f"imgsz={s} {name:10s}: box short side p10 = {np.percentile(short[m] * s, 10):.0f} px, median = {np.median(short[m] * s):.0f} px")

## 1.2 Clean, split and convert to YOLO format
- **Clean:** clip every box to the image. Then drop component boxes with a side under 5 px, and text boxes with a side of 1 px or less (texts are naturally thin, so they get a looser rule).
- **Classes:** the 17 component classes keep ids 0–16; `text` is id 17 (`TEXT_CLS`).
- **Split by image, not by box**, so the same drawing never appears in train and test (no leakage). Stratify on each image's *rarest component* class (text is in almost every image, so it is left out of the key). Keys with fewer than 10 images are merged into the largest group: with a 80/10/10 split done in two steps, each key needs at least 2 images in the temporary 20 % part, i.e. at least 10 images overall. Images without a valid component box are left out (the count is printed). *Caveat:* if the dataset contains several photos or scans of the same circuit, they are different images for this split, so some leakage can remain.
- **One split for the whole pipeline:** the reader in Part 2 and the matching in Part 3 reuse it.
- **YOLO format:** one `.txt` per image with both components and text, `cls xc yc w h` normalized to 0-1.
- The output folder is deleted first, so files from a previous run cannot end up in the wrong split.
- The crosstab at the end checks that every class appears in train, val and test.

In [ ]:
# Clip to the image, then drop degenerate boxes
boxes["x1"], boxes["y1"] = boxes["x1"].clip(lower=0), boxes["y1"].clip(lower=0)
boxes["x2"], boxes["y2"] = np.minimum(boxes["x2"], boxes["W"]), np.minimum(boxes["y2"], boxes["H"])
bw, bh, is_text = boxes["x2"] - boxes["x1"], boxes["y2"] - boxes["y1"], boxes["class"] == "text"
keep = np.where(is_text, (bw > 1) & (bh > 1), (bw >= 5) & (bh >= 5))
print("Removed degenerate boxes:", boxes[~keep]["class"].eq("text").map({True: "text", False: "components"}).value_counts().to_dict())
boxes = boxes[keep].copy()

# Class ids: components 0..16 (sorted COCO ids), text 17
cat_ids = sorted(cats)
names = [cats[c] for c in cat_ids] + ["text"]
TEXT_CLS = len(names) - 1
boxes["cls"] = boxes["class"].map({n: i for i, n in enumerate(names)})

# Stratified split by image, on the rarest component class
comp_boxes = boxes[boxes["cls"] != TEXT_CLS]
class_freq = comp_boxes["cls"].value_counts()
img_key = comp_boxes.groupby("image_id")["cls"].apply(lambda s: min(s, key=lambda c: class_freq[c]))
key_counts = img_key.value_counts()
small = key_counts[key_counts < 10].index
img_key[img_key.isin(small)] = key_counts.idxmax()
print(f"Images left out (no valid component box): {len(imgs) - len(img_key)}")

all_ids = img_key.index.values
train_ids, tmp_ids = train_test_split(all_ids, test_size=0.2, random_state=SEED, stratify=img_key[all_ids])
val_ids, test_ids  = train_test_split(tmp_ids, test_size=0.5, random_state=SEED, stratify=img_key[tmp_ids])
splits = {"train": train_ids, "val": val_ids, "test": test_ids}
print({k: len(v) for k, v in splits.items()})
boxes["split"] = boxes["image_id"].map({i: s for s, ids in splits.items() for i in ids})
boxes = boxes[boxes["split"].notna()]

# Write the files in YOLO format (xc, yc, w, h), components and text together
shutil.rmtree(OUT, ignore_errors=True)
by_image = dict(tuple(boxes.groupby("image_id")))
for sp, ids in splits.items():
    os.makedirs(f"{OUT}/images/{sp}"); os.makedirs(f"{OUT}/labels/{sp}")
    for iid in ids:
        fname = imgs.loc[iid, "file_name"]
        W, H = imgs.loc[iid, "width"], imgs.loc[iid, "height"]
        shutil.copy(os.path.join(IMG_DIR, fname), f"{OUT}/images/{sp}/{fname}")
        g = by_image[iid]
        lines = [f"{c} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}"
                 for c, x1, y1, x2, y2 in zip(g["cls"], g["x1"], g["y1"], g["x2"], g["y2"])]
        open(f"{OUT}/labels/{sp}/{os.path.splitext(fname)[0]}.txt", "w").write("\n".join(lines))

yaml.safe_dump({"path": OUT, "train": "images/train", "val": "images/val", "test": "images/test",
                "names": dict(enumerate(names))}, open(DATA_YAML, "w"), sort_keys=False)

# Check that every class appears in every split
counts = pd.crosstab(boxes["class"], boxes["split"])[["train", "val", "test"]]
print(counts.sort_values("train", ascending=False))
split_df = pd.DataFrame([(i, s) for s, ids in splits.items() for i in ids], columns=["image_id", "split"])
split_df["file_name"] = split_df["image_id"].map(imgs["file_name"])
split_df.to_csv(f"{RUNS}/split.csv", index=False)

# Tables used by the reader (Part 2) and the matching (Part 3)
BOX = ["x1", "y1", "x2", "y2"]
boxes["stem"] = boxes["image_id"].map(imgs["stem"])
text_df = boxes[(boxes["cls"] == TEXT_CLS) & boxes["stem"].isin(stem2file)][["stem", "text", *BOX, "split"]]
comp_df = boxes[boxes["cls"] != TEXT_CLS][["stem", "class", *BOX, "split"]].rename(columns={"class": "cls"})
print("vertical text boxes (taller than wide):", round(float(((text_df.y2 - text_df.y1) > (text_df.x2 - text_df.x1)).mean()), 4))

## 1.3 Visual check of the conversion
Draw the YOLO labels back on random training images. Boxes must sit on the components and on the handwritten values, with the right class name; if they do not, the conversion is wrong.

In [ ]:
def draw_gt(img, lbl_path, color=(0, 170, 0)):
    H, W = img.shape[:2]
    for line in open(lbl_path).read().strip().splitlines():
        c, xc, yc, w, h = line.split(); c = int(c); xc, yc, w, h = map(float, (xc, yc, w, h))
        p1, p2 = (int((xc-w/2)*W), int((yc-h/2)*H)), (int((xc+w/2)*W), int((yc+h/2)*H))
        cv2.rectangle(img, p1, p2, color, 3)
        cv2.putText(img, names[c], (p1[0], max(p1[1]-6, 15)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
    return img

def label_of(path):
    return os.path.splitext(path.replace("/images/", "/labels/"))[0] + ".txt"

sample = random.sample(glob.glob(f"{OUT}/images/train/*"), 4)     # any extension (dataset is .jpg)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, path in zip(axes.ravel(), sample):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    ax.imshow(draw_gt(img, label_of(path))); ax.set_title(os.path.basename(path)); ax.axis("off")
plt.tight_layout(); plt.show()

## 1.4 Train the detector
| Setting | Value | Reason |
|---|---|---|
| model | `yolo11m.pt` (COCO pretrained) | Transfer learning works well with a small dataset; the medium size keeps fine details (e.g. arrow direction) and trains in about 2 h on a T4. One model for components and text: one training, one pass per image. |
| classes | 17 components + `text` | The text boxes come from the same images, so they are labelled together. |
| imgsz | 1024 | Symbols and texts stay large enough after resizing (see box sizes in 1.1). |
| epochs / patience | 100 / 20 | Enough to converge; training stops early if validation does not improve for 20 epochs. |
| batch | 4 | Fits a 16 GB T4 at 1024 px. |
| fliplr | 0 | Mirrored drawings / handwriting do not occur in real data. |
| degrees | 5 | Scanned or photographed drawings are often slightly tilted. |
| seed | 42 | Reproducibility. |

In [ ]:
model = YOLO("yolo11m.pt")
model.train(data=DATA_YAML, imgsz=1024, epochs=100, patience=20, batch=4, device=0,
            fliplr=0.0, degrees=5, seed=SEED, workers=4,
            project=RUNS, name=RUN_NAME, exist_ok=False, plots=True)
BEST = f"{model.trainer.save_dir}/weights/best.pt"
del model; torch.cuda.empty_cache()     # free GPU memory before TrOCR
print(BEST)

## 1.5 Evaluate on the validation set
- **mAP50 and mAP50-95:**
  - mAP50: a box is correct if it overlaps the true box by at least 50 % (IoU ≥ 0.5).
  - mAP50-95: the same, averaged over stricter overlaps up to 95 %, so it also measures how tight the boxes are.
  - We validate with `conf=0.001` so the model keeps all its boxes, even the unsure ones; mAP needs the full precision-recall curve.
- **Components and text are reported separately.** Ultralytics' overall mAP would average `text` in as an 18th class; instead the component mAP is the mean over the 17 component classes (comparable with a components-only detector), and text gets its own AP.
- **Two thresholds, both chosen on validation:** `CONF` is the confidence with the best mean F1 over the component classes; `TEXT_CONF` the one with the best F1 for `text`. They are reused unchanged on the test set and in the full pipeline.
- **P and R** per class come from Ultralytics' best-F1 point over all classes, which can differ slightly from either threshold.
- **Confusion matrix:** computed in a second validation run at `CONF` (the component threshold, so the `text` row/column is slightly stricter than in the pipeline). At 0.001 the matrix would be full of low-confidence false alarms.
- **`n_gt`:** number of true boxes per class in that split. A class with few boxes gives a less reliable score.

In [ ]:
model = YOLO(BEST)

def evaluate(split, conf=None, text_conf=None):
    m = model.val(data=DATA_YAML, split=split, imgsz=1024, batch=8, conf=0.001, device=0,
                  project=RUNS, name=split, exist_ok=True)
    idx = np.array(m.box.ap_class_index)
    if conf is None:                                   # choose the thresholds (validation only)
        f1 = m.box.f1_curve                            # one row per class in idx
        grid = np.linspace(0, 1, f1.shape[1])
        conf = float(grid[f1[idx != TEXT_CLS].mean(0).argmax()])
        text_conf = float(grid[f1[idx == TEXT_CLS].mean(0).argmax()])
    per_class = pd.DataFrame({
        "class": [names[c] for c in idx],
        "n_gt":  [counts.loc[names[c], split] for c in idx],
        "P": m.box.p, "R": m.box.r, "AP50": m.box.ap50, "AP50-95": m.box.ap,
    }).sort_values("AP50-95")
    comp, text = per_class[per_class["class"] != "text"], per_class[per_class["class"] == "text"].iloc[0]
    overall = {"components": {"mAP50": float(comp["AP50"].mean()), "mAP50-95": float(comp["AP50-95"].mean()),
                              "P": float(comp["P"].mean()), "R": float(comp["R"].mean()), "conf": conf},
               "text": {"AP50": float(text["AP50"]), "AP50-95": float(text["AP50-95"]),
                        "P": float(text["P"]), "R": float(text["R"]), "conf": text_conf}}
    m_cm = model.val(data=DATA_YAML, split=split, imgsz=1024, batch=8, conf=conf, device=0,
                     project=RUNS, name=f"{split}_cm", exist_ok=True)
    return per_class, overall, m_cm.confusion_matrix.matrix

pc_val, ov_val, cm_val = evaluate("val")
CONF, TEXT_CONF = ov_val["components"]["conf"], ov_val["text"]["conf"]
print(json.dumps(ov_val, indent=2)); print(pc_val.round(3).to_string(index=False))
pc_val.round(4).to_csv(f"{RUNS}/val_per_class_metrics.csv", index=False)

## 1.6 Error analysis (validation)
- **Confusion pairs:** a true class predicted as another class (including component ↔ text).
- **Missed objects:** true boxes with no detection (false negatives).
- **False alarms:** detections with no true object (false positives).

Ultralytics stores the matrix as `matrix[predicted, true]`; the last index is *background*. The same analysis is repeated on the test set in Part 4. The figure shows predictions (model colours) with the ground truth drawn on top in **green**, so missed, extra and misclassified boxes are visible.

In [ ]:
def error_analysis(cm):
    nc = len(names)
    pairs = [(names[t], names[p], int(cm[p, t])) for t in range(nc) for p in range(nc) if p != t and cm[p, t] > 0]
    print("Top class confusions (true -> predicted):")
    for t, p, n in sorted(pairs, key=lambda x: -x[2])[:10]: print(f"  {t:20s} -> {p:20s} {n}")
    missed = {names[t]: int(cm[nc, t]) for t in range(nc)}         # true object, no detection
    false_alarm = {names[p]: int(cm[p, nc]) for p in range(nc)}    # detection, no object
    print("\nMissed per class:", dict(sorted(missed.items(), key=lambda x: -x[1])[:6]))
    print("False alarms per class:", dict(sorted(false_alarm.items(), key=lambda x: -x[1])[:6]))

def show_predictions(split, n=4):
    sample = random.sample(glob.glob(f"{OUT}/images/{split}/*"), n)
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    for ax, path in zip(axes.ravel(), sample):
        r = model.predict(path, imgsz=1024, conf=min(CONF, TEXT_CONF), device=0, verbose=False)[0]
        k, s = r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()
        keep = ((k == TEXT_CLS) & (s >= TEXT_CONF)) | ((k != TEXT_CLS) & (s >= CONF))   # each group at its own threshold
        r = r.cpu()[torch.from_numpy(keep)]
        img = cv2.cvtColor(r.plot(line_width=2, font_size=10), cv2.COLOR_BGR2RGB)
        ax.imshow(draw_gt(img, label_of(path)))
        ax.set_title(f"{os.path.basename(path)}: {len(r.boxes)} detections"); ax.axis("off")
    plt.tight_layout(); plt.show()

error_analysis(cm_val)
show_predictions("val")

# Part 2: Reading the values (fine-tune TrOCR)
The detector finds *where* the text is; this part teaches the reader *what* it says. It is trained and scored on crops of the ground-truth text boxes, so its errors are separated from detection errors. The full chain (detected boxes → reader) is evaluated in Part 3.

## 2.1 Crops and TrOCR without training
Each text box is cropped from the image (with 4 px of margin) and read by TrOCR. Plain boxes, no rotation: only about 0.5 % of the labels are vertical (printed in 1.2), and the detector also returns plain boxes, so training and use see the same kind of crop.

**Metrics:** exact match (the whole string is right) and CER (character error rate: edits needed to fix the prediction, divided by the true length).

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
OCR_ID = "microsoft/trocr-base-handwritten"
processor = TrOCRProcessor.from_pretrained(OCR_ID)
tok = processor.tokenizer

def norm(t):
    return t.replace(" ", "").replace("µ", "μ").replace("Ω", "Ω")   # micro sign -> mu, ohm sign -> omega

def load_rgb(s):
    return cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, stem2file[s])), cv2.COLOR_BGR2RGB)

def crop(img, b, pad=4):
    H, W = img.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in b]
    return img[max(y1 - pad, 0):min(y2 + pad, H), max(x1 - pad, 0):min(x2 + pad, W)].copy()

def make_crops(split):
    crops, labels = [], []
    for s, t in text_df[text_df.split == split].groupby("stem"):
        img = load_rgb(s)
        crops += [crop(img, b) for b in t[BOX].values]
        labels += list(t.text)
    return crops, labels

crops = {s: make_crops(s) for s in ("train", "val", "test")}
print({s: len(v[0]) for s, v in crops.items()})

def read_crops(images, bs=32):
    ocr.eval(); out = []
    for i in range(0, len(images), bs):
        pv = processor(images=[Image.fromarray(c) for c in images[i:i + bs]], return_tensors="pt").pixel_values.to(DEVICE)
        with torch.no_grad(), torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            ids = ocr.generate(pv, max_new_tokens=16)
        out += [norm(t) for t in processor.batch_decode(ids, skip_special_tokens=True)]
    return out

def edit_distance(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[len(b)]

def read_metrics(pred, gold):
    gold = [norm(g) for g in gold]
    return {"exact match": round(np.mean([p == g for p, g in zip(pred, gold)]), 3),
            "CER": round(sum(edit_distance(p, g) for p, g in zip(pred, gold)) / sum(len(g) for g in gold), 3)}

ocr = VisionEncoderDecoderModel.from_pretrained(OCR_ID).to(DEVICE)
pred = read_crops(crops["val"][0])
print("zero-shot on val:", read_metrics(pred, crops["val"][1]))
print("examples (pred | true):", list(zip(pred[:15], crops["val"][1][:15])))

## 2.2 Fine-tune TrOCR on the training crops
3 epochs on the ~9,500 training crops (about 8 minutes each on a T4). The loss is computed here explicitly: with transformers 4.57 the built-in loss of this model shifts the labels a second time (the warning `loss_type=None ... ForCausalLMLoss`), which trains the model to predict two characters ahead. After each epoch the reader is scored on the validation crops and the best epoch is kept (decision on validation).

In [ ]:
EPOCHS, BS, LR = 3, 16, 5e-5
from transformers.models.vision_encoder_decoder.modeling_vision_encoder_decoder import shift_tokens_right
if ocr.config.decoder_start_token_id is None: ocr.config.decoder_start_token_id = tok.cls_token_id
if ocr.config.pad_token_id is None: ocr.config.pad_token_id = tok.pad_token_id

class CropDS(torch.utils.data.Dataset):
    def __init__(self, images, labels): self.images, self.labels = images, labels
    def __len__(self): return len(self.images)
    def __getitem__(self, i):
        pv = processor(images=Image.fromarray(self.images[i]), return_tensors="pt").pixel_values[0]
        return pv, torch.tensor(tok(norm(self.labels[i]), max_length=24, truncation=True).input_ids)

def collate(batch):
    pv = torch.stack([b[0] for b in batch])
    labels = torch.nn.utils.rnn.pad_sequence([b[1] for b in batch], batch_first=True, padding_value=-100)
    return pv, labels

loader = torch.utils.data.DataLoader(CropDS(*crops["train"]), batch_size=BS, shuffle=True,
                                     num_workers=2, collate_fn=collate, generator=torch.Generator().manual_seed(SEED))
opt = torch.optim.AdamW(ocr.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda")
best_em, history = -1, []
for epoch in range(1, EPOCHS + 1):
    ocr.train(); t0 = datetime.datetime.now(); losses = []
    for pv, labels in loader:
        labels = labels.to(DEVICE)
        dec_in = shift_tokens_right(labels, ocr.config.pad_token_id, ocr.config.decoder_start_token_id)  # shift once
        with torch.autocast("cuda", dtype=torch.float16):
            logits = ocr(pixel_values=pv.to(DEVICE), decoder_input_ids=dec_in).logits
        loss = torch.nn.functional.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                                                 labels.reshape(-1), ignore_index=-100)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        losses.append(loss.item())
    res = read_metrics(read_crops(crops["val"][0]), crops["val"][1])
    history.append({"epoch": epoch, "train loss": round(np.mean(losses), 4), **res})
    print(history[-1], "| time:", datetime.datetime.now() - t0)
    if res["exact match"] > best_em:
        best_em = res["exact match"]; ocr.save_pretrained(OCR_DIR); processor.save_pretrained(OCR_DIR)
del opt, scaler, loader, ocr; torch.cuda.empty_cache()     # free GPU memory for the YOLO models

ocr = VisionEncoderDecoderModel.from_pretrained(OCR_DIR).to(DEVICE)   # best epoch on validation
pred_val = read_crops(crops["val"][0])
read_val = read_metrics(pred_val, crops["val"][1])
print("fine-tuned on val:", read_val)
errors = [(p, norm(g)) for p, g in zip(pred_val, crops["val"][1]) if p != norm(g)]
print("some errors (pred | true):", errors[:20])

# Part 3: Matching and full pipeline

## 3.1 Matching rule, checked on the ground truth
The unit says which classes are possible: `Ω` → Resistor, `F` → Capacitor, `H` → Inductor, `Hz` or `sin/cos` → AC sources, `V` → voltage sources, `A` → current sources (`Hz` is checked before `H`). The text goes to the nearest possible component, measured as the gap between the two boxes. A text without a unit goes to the nearest component that can carry a value. Wire crossovers, diodes, transistors, op-amps and GND never get a value.

There is no text-component link in the dataset, so the matching cannot be scored directly. Instead, on ground-truth boxes (no detection or reading errors), texts that have a unit are matched by distance only (nearest component that can carry a value), and we check whether the chosen component's class fits the unit. This measures the geometric part of the rule at class level (not which of two neighbours of the same class), and also shows how often the unit rule changes the choice.

In [ ]:
UNIT_CLASSES = {"AC": {"V-AC", "I-AC"}, "Hz": {"V-AC", "I-AC"}, "Ω": {"Resistor"}, "F": {"Capacitor"},
                "H": {"Inductor"}, "V": {"V-DC", "V-DC (one port)", "V-AC"}, "A": {"I-DC", "I-AC"}}
VALUE_CLASSES = set().union(*UNIT_CLASSES.values())
DEFAULT_UNIT = {"Resistor": "Ω", "Capacitor": "F", "Inductor": "H", "V-DC": "V", "V-DC (one port)": "V", "I-DC": "A"}

def unit_of(text):
    t = norm(text)
    if "sin" in t or "cos" in t:
        return "AC"
    for u in ("Hz", "Ω", "F", "H", "V", "A"):
        if u in t:
            return u
    return None

def with_unit(text, cls):       # "1k" next to a resistor -> "1kΩ"
    return text if unit_of(text) or cls not in DEFAULT_UNIT else text + DEFAULT_UNIT[cls]

def box_gap(a, b):              # distance between two boxes, 0 if they touch
    dx = max(b[0] - a[2], a[0] - b[2], 0)
    dy = max(b[1] - a[3], a[1] - b[3], 0)
    return float(np.hypot(dx, dy))

def match_image(texts, comps, use_units=True):
    """texts: [(text, box)], comps: [(class, box)] -> index of the chosen component for each text (or None)"""
    out = []
    for text, tb in texts:
        u = unit_of(text) if use_units else None
        allowed = UNIT_CLASSES[u] if u else VALUE_CLASSES
        cands = [(box_gap(tb, cb), j) for j, (c, cb) in enumerate(comps) if c in allowed]
        out.append(min(cands)[1] if cands else None)
    return out

comp_by = dict(tuple(comp_df.groupby("stem")))

def matcher_check(split):
    """Match ground-truth texts to ground-truth components, with and without the unit rule."""
    rows = []
    for s, t in text_df[text_df.split == split].groupby("stem"):
        c = comp_by.get(s, comp_df.iloc[:0])
        comps = list(zip(c.cls, c[BOX].values))
        texts = list(zip(t.text, t[BOX].values))
        for (text, _), a, b in zip(texts, match_image(texts, comps), match_image(texts, comps, use_units=False)):
            rows.append(dict(unit=unit_of(text), attached=a is not None,
                             dist_cls=comps[b][0] if b is not None else None, changed=a != b))
    r = pd.DataFrame(rows)
    u = r[r.unit.notna()]
    return {"texts": len(r), "with unit": round(len(u) / len(r), 3),
            "attached": round(r.attached.mean(), 3),
            "distance-only class fits unit": round(np.mean([d in UNIT_CLASSES[k] for d, k in zip(u.dist_cls, u.unit)]), 3),
            "unit changed the choice": round(r.changed.mean(), 3)}

ms_val = matcher_check("val")
print(ms_val)

## 3.2 Full pipeline on validation
`read_circuit(path)` runs the detector once, keeps components above `CONF` and text above `TEXT_CONF` (both from 1.5), reads each text box with TrOCR and applies the matching rule. Evaluated on all validation images against the ground-truth text:
- **found:** a predicted text box overlaps the true one (IoU ≥ 0.5);
- **found and read:** and the text is read exactly;
- **attached:** and it was given to a component (of a class that fits the unit; whether it is the *right* component is not checked, see 3.1).

These are rates over the true texts (recall): extra predicted boxes are not penalised.

In [ ]:
def read_circuit(path, comp_conf=None, text_conf=None):
    """-> components [(class, box)], texts [dict(text, text_box, cls, comp_box, value)]"""
    comp_conf = CONF if comp_conf is None else comp_conf
    text_conf = TEXT_CONF if text_conf is None else text_conf
    bgr = cv2.imread(str(path))                     # YOLO expects BGR arrays
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)      # TrOCR crops in RGB
    b = model.predict(bgr, imgsz=1024, conf=min(comp_conf, text_conf), device=0, verbose=False)[0].boxes
    k, s, xyxy = b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(), b.xyxy.cpu().numpy()
    comps = [(names[c], box) for c, sc, box in zip(k, s, xyxy) if c != TEXT_CLS and sc >= comp_conf]
    tb = xyxy[(k == TEXT_CLS) & (s >= text_conf)]
    texts = read_crops([crop(img, t) for t in tb]) if len(tb) else []
    links = match_image(list(zip(texts, tb)), comps)
    rows = [dict(text=t, text_box=box,
                 cls=comps[j][0] if j is not None else None,
                 comp_box=comps[j][1] if j is not None else None,
                 value=with_unit(t, comps[j][0]) if j is not None else None)
            for t, box, j in zip(texts, tb, links)]
    return comps, rows

def iou_matrix(a, b):
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    tl, br = np.maximum(a[:, None, :2], b[None, :, :2]), np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def end_to_end(split):
    n = found = read_ok = attached = 0
    for s, gt in text_df[text_df.split == split].groupby("stem"):
        _, rows = read_circuit(os.path.join(IMG_DIR, stem2file[s]))
        iou = iou_matrix(gt[BOX].values, [r["text_box"] for r in rows])
        for i, g in enumerate(gt.text):
            n += 1
            if iou.shape[1] and iou[i].max() >= 0.5:
                r = rows[iou[i].argmax()]; found += 1
                if r["text"] == norm(g):
                    read_ok += 1; attached += r["cls"] is not None
    return {"texts": n, "found": round(found / n, 3), "found and read": round(read_ok / n, 3),
            "found, read and attached": round(attached / n, 3)}

e2e_val = end_to_end("val")
print(e2e_val)

# Part 4: Final evaluation on the test set (run once)

Runs only with `RUN_TEST = True`, after every choice is fixed on validation (detector weights, `CONF`, `TEXT_CONF`, reader epoch). `TEST_DONE.txt` is written as soon as the test set is touched, and section 0 refuses to start a new test run while it exists.

The detector makes two passes over the test images (one for mAP, one for the confusion matrix), with the same weights and no tuning in between. Value reading uses the same measurements as on validation.

In [ ]:
pc_test = ov_test = cm_test = values_test = None
if RUN_TEST:
    open(TEST_DONE, "w").write(f"{datetime.datetime.now()} conf={CONF:.3f} text_conf={TEXT_CONF:.3f}")

    # Detection: components and text
    pc_test, ov_test, cm_test = evaluate("test", conf=CONF, text_conf=TEXT_CONF)
    print(json.dumps(ov_test, indent=2)); print(pc_test.round(3).to_string(index=False))
    pc_test.round(4).to_csv(f"{RUNS}/test_per_class_metrics.csv", index=False)
    error_analysis(cm_test)
    show_predictions("test")

    # Reading and full pipeline
    values_test = {"matcher (ground truth)": matcher_check("test"),
                   "reader (ground-truth crops)": read_metrics(read_crops(crops["test"][0]), crops["test"][1]),
                   "end to end": end_to_end("test")}
    print(json.dumps(values_test, indent=2, ensure_ascii=False))
else:
    print("RUN_TEST = False: test set not evaluated")

# Part 5: Examples and outputs

## 5.1 Example
Components in green with their class; each value in red at its text, with a line to the component it was attached to. The reader only knows the dataset's labelling style (values without designators), so a label like `R1 4.7kΩ` is misread.

In [ ]:
def show_reading(path, save_as=None, **kw):
    comps, rows = read_circuit(path, **kw)
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(16, 11)); ax.imshow(img); ax.axis("off")    # matplotlib can draw Ω and μ, cv2 cannot
    for cls, (x1, y1, x2, y2) in comps:
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="green", lw=2))
        ax.text(x1, y1 - 6, cls, color="green", fontsize=12)
    for r in rows:
        if r["value"]:
            (tx1, ty1, tx2, ty2), (cx1, cy1, cx2, cy2) = r["text_box"], r["comp_box"]
            ax.plot([(tx1 + tx2) / 2, (cx1 + cx2) / 2], [(ty1 + ty2) / 2, (cy1 + cy2) / 2], color="red", lw=1)
            ax.text(tx1, ty1 - 6, r["value"], color="red", fontsize=14, fontweight="bold")
    if save_as:
        fig.savefig(save_as, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"{len(comps)} components, {len(rows)} texts read, {sum(r['value'] is not None for r in rows)} attached")

path = os.path.join(IMG_DIR, stem2file[text_df[text_df.split == "val"].stem.iloc[0]])
show_reading(path, save_as=f"{RUNS}/example_values.png")

## 5.2 Try your own drawing
Qualitative check on a circuit that is not part of the dataset (attach it as `circuit0.jpg`, or change the pattern). It is only a visual sanity check, not a metric. For drawings or photos that do not look like the scans, a lower component threshold such as `comp_conf=0.25` helps.

In [ ]:
found = glob.glob("/kaggle/input/**/circuit0.jpg", recursive=True)
if not found:
    print("Own image not attached: skipping this cell")
else:
    show_reading(found[0], comp_conf=0.25)

## 5.3 Save outputs
One zip, `circuit-reader-outputs.zip`, with the detector (weights, curves, confusion matrices), the split, all metrics, the package versions, the example figure and the test record. Files that only exist after the test run are included only if present, so the cell also works with `RUN_TEST = False`. The fine-tuned reader stays in `/kaggle/working/trocr_values` (about 1.3 GB, download only if needed).

In [ ]:
from IPython.display import FileLink
json.dump({"val": ov_val, "test": ov_test}, open(f"{RUNS}/detection_summary.json", "w"), indent=2)
values_summary = {"val": {"matcher (ground truth)": ms_val, "reader (ground-truth crops)": read_val,
                          "reader history": history, "end to end": e2e_val},
                  "test": values_test}
json.dump(values_summary, open(f"{RUNS}/values_summary.json", "w"), indent=2, ensure_ascii=False, default=float)
os.system(f"pip freeze > {RUNS}/requirements.txt")
if os.path.exists(TEST_DONE):
    shutil.copy(TEST_DONE, RUNS)
print(json.dumps({"detection": {"val": ov_val, "test": ov_test}, "values": values_summary},
                 indent=2, ensure_ascii=False, default=float))

shutil.make_archive(f"{WORK}/circuit-reader-outputs", "zip", RUNS)
FileLink("circuit-reader-outputs.zip")